# Question 2

In [1]:
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, hamming_loss, f1_score

In [2]:
# Load iris
iris = load_iris()
X = iris.data
y_class = iris.target

# Tạo multi-label Y (4 labels)
petal_length = X[:, 2]
sepal_width = X[:, 1]

Y = np.column_stack([
    (y_class == 0).astype(int),
    (y_class == 2).astype(int),
    (petal_length > np.median(petal_length)).astype(int),
    (sepal_width  > np.median(sepal_width)).astype(int),
])
label_names = ["is_setosa", "is_virginica", "petal_long", "sepal_wide"]

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=42, stratify=y_class
)

L = Y.shape[1]

In [4]:
from sklearn.multioutput import ClassifierChain

chain_params = {"order": "random", "random_state": 42}
# 4 classifier
models = {
    "LogisticRegression (CC)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", ClassifierChain(LogisticRegression(max_iter=2000), **chain_params))
    ]),

    "SVM-RBF (CC)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", ClassifierChain(SVC(kernel="rbf", probability=True, gamma="scale"), **chain_params))
    ]),

    "RandomForest (CC)": Pipeline([
        # Random Forest khong bat buoc phai co Scaler
        ("clf", ClassifierChain(RandomForestClassifier(n_estimators=300, random_state=42), **chain_params))
    ]),

    "KNN (CC)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", ClassifierChain(KNeighborsClassifier(n_neighbors=7), **chain_params))
    ])
}

In [5]:
def evaluate(model, name):
    Y_pred = model.predict(X_test)

    subset_acc = accuracy_score(Y_test, Y_pred)
    ham = hamming_loss(Y_test, Y_pred)
    f1_micro = f1_score(Y_test, Y_pred, average="micro", zero_division=0)
    f1_macro = f1_score(Y_test, Y_pred, average="macro", zero_division=0)

    print(f"\n=== {name} ===")
    print(f"Labels: {label_names}")
    print(f"Subset accuracy (exact match): {subset_acc:.4f}")

    ovr = model.named_steps["clf"]
    print(f"# binary classifiers inside this model: {len(ovr.estimators_)} (should be {L})")


In [6]:
# Train
total_binary = 0
for name, model in models.items():
    model.fit(X_train, Y_train)
    evaluate(model, name)
    total_binary += len(model.named_steps["clf"].estimators_)

print(f"\nTOTAL binary classifiers trained across 4 different models: {total_binary} (expected {4*L})")


=== LogisticRegression (CC) ===
Labels: ['is_setosa', 'is_virginica', 'petal_long', 'sepal_wide']
Subset accuracy (exact match): 0.8222
# binary classifiers inside this model: 4 (should be 4)

=== SVM-RBF (CC) ===
Labels: ['is_setosa', 'is_virginica', 'petal_long', 'sepal_wide']
Subset accuracy (exact match): 0.8667
# binary classifiers inside this model: 4 (should be 4)


/home/huyduong/anaconda3/envs/aio26/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/huyduong/anaconda3/envs/aio26/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/huyduong/anaconda3/envs/aio26/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/huyduong/anaconda3/envs/aio26/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The


=== RandomForest (CC) ===
Labels: ['is_setosa', 'is_virginica', 'petal_long', 'sepal_wide']
Subset accuracy (exact match): 0.8889
# binary classifiers inside this model: 4 (should be 4)

=== KNN (CC) ===
Labels: ['is_setosa', 'is_virginica', 'petal_long', 'sepal_wide']
Subset accuracy (exact match): 0.7778
# binary classifiers inside this model: 4 (should be 4)

TOTAL binary classifiers trained across 4 different models: 16 (expected 16)


# Question 3


In [ ]:
#!pip install scikit-multilearn

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, hamming_loss

from skmultilearn.problem_transform import LabelPowerset

In [2]:
# Load iris
iris = load_iris()
X = iris.data
y_class = iris.target

# Tạo multi-label Y (4 labels)
petal_length = X[:, 2]
sepal_width = X[:, 1]

Y = np.column_stack([
    (y_class == 0).astype(int),
    (y_class == 2).astype(int),
    (petal_length > np.median(petal_length)).astype(int),
    (sepal_width  > np.median(sepal_width)).astype(int),
])
label_names = ["is_setosa", "is_virginica", "petal_long", "sepal_wide"]

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=42, stratify=y_class
)

In [4]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Insert your code here
classifier = LabelPowerset(classifier=LogisticRegression(max_iter=2000), require_dense=[True, True])
classifier.fit(X_train_scaled, Y_train)

predictions_sparse = classifier.predict(X_test_scaled)
predictions = predictions_sparse.toarray()

print("Subset Accuracy:", accuracy_score(Y_test, predictions))

Subset Accuracy: 0.7777777777777778
